# Exercice n°8: Visualisation et interprétation des modèles

Cet exercice vise à tester vos compétences en visualisation de données en python et son application à l'interprétation de modèles d'apprentissage machine.

**Mise en contexte**

Vous effectuez un stage de recherche dans un laboratoire recherche. Dans le cadre de ce stage, votre superviseure vous demande de vous familiariser avec les données phénotypiques et les données d'IRM fonctionnelle au repos du jeu de données **ADHD200** . Vous devrez également l'aider à interpréter un modèle d'apprentissage machine prédisant le groupe diagnostique grâce à la visualisation.

**Consignes importantes pour l'évaluation automatique :**

1. **Conformité** : Ne modifiez pas les noms des variables de résultat (ex: q1_n_sujets, defi_accuracy) donnés dans les lignes commentées. Ces noms sont essentiels pour l'évaluation automatique.
2. **Exécution** : Assurez-vous que votre code s'exécute sans erreur de haut en bas.
Paramètres : Respectez exactement les paramètres spécifiés dans chaque question. L'évaluateur automatique compare vos résultats à des valeurs de référence calculées avec ces paramètres précis.
3. **Données** : Utilisez uniquement les variables pheno, func et confounds fournies dans la cellule de configuration (Section 0), qui sont déjà alignées entre elles.

Bon courage !

## Section 0: Configuration et chargement des données

Exécutez ces cellules de configuration. Elles téléchargent les données et préparent les variables que vous utiliserez dans l'exercice. **Ne les modifiez pas.**

In [ ]:
# Configuration — ne pas modifier
import warnings
warnings.filterwarnings('ignore')

import os
import statsmodels
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from cmcrameri import cm
from nilearn import datasets, plotting
from nilearn.image import math_img
from nilearn.maskers import NiftiLabelsMasker
from matplotlib.colors import ListedColormap
from nilearn.connectome import ConnectivityMeasure

print("Bibliothèques importées avec succès.")

In [ ]:
# Téléchargement des données pour l'exercice
!wget -nc https://raw.githubusercontent.com/psy3019-6973/h2026-exercises/main/psy3019_Exercise_8/exercice_ml_coefficients.npz
!wget -nc https://raw.githubusercontent.com/psy3019-6973/h2026-exercises/main/psy3019_Exercise_8/exercice_ml_timeseries.npz

In [ ]:
# Chargement du dataset ADHD200 — ne pas modifier
data_dir = './nilearn_data'

adhd_dataset = datasets.fetch_adhd(n_subjects=40, data_dir=data_dir)

# Les données phénotypiques brutes (peuvent ne pas couvrir tous les sujets)
pheno_brut = adhd_dataset.phenotypic

print(f"Dataset chargé : {len(adhd_dataset.func)} images fonctionnelles")
print(f"Données phénotypiques brutes : {pheno_brut.shape[0]} sujets")

In [ ]:
# Alignement des données — ne pas modifier
#
# Certains sujets ont des images IRMf mais pas de données phénotypiques.
# On conserve uniquement les sujets présents dans les deux sources.

func_ids = [int(os.path.basename(f).split('_')[0]) for f in adhd_dataset.func]
pheno_ids = set(pheno_brut['Subject'].values)
matched_idx = [i for i, fid in enumerate(func_ids) if fid in pheno_ids]
matched_fids = [func_ids[i] for i in matched_idx]

# Données alignées et prêtes à l'emploi
pheno    = pheno_brut.set_index('Subject').loc[matched_fids].reset_index()
func     = [adhd_dataset.func[i] for i in matched_idx]
confounds = [adhd_dataset.confounds[i] for i in matched_idx]

print(f"Sujets avec données d'imagerie ET phénotypiques : {len(func)}")
print(f"Colonnes phénotypiques disponibles : {list(pheno.columns)}")

# Partie 1 : Exploration des données phénotypiques

Dans cette section, vous allez explorer les variables cliniques et démographiques du jeu de données ADHD200.

### Question 1: Distribution de l'âge selon le sexe

Visualisez la variable `age` dans le *dataframe* `pheno` en fonction de la variable `sex`. Remplissez la fonction `histplot` ci-dessous en précisant les bonnes variables. Vous devrez préciser les paramètres suivants:

- Spécifiez la structure de données (dataframe) à utiliser dans la fonction.
- La variable `age` doit figurée sur l'axe des x.
- Les distributions de l'âge doivent être séparées selon la variable `sex`. Cela doit être spécifié dans la même fonction `histplot` ci-dessous.
- Superposez une estimation de la densité par noyau (`kdeplot`) à l'histogramme.

Vous devrez donc spécifier la valeur de quatre paramètes. **Laisser les valeurs par défaut pour les autres variables de la fonction.**

Au besoin, référez-vous à la documentation de [`seaborn`](https://seaborn.pydata.org/generated/seaborn.histplot.html).

In [ ]:
ax_q1 = plt.subplot()

sns.histplot(
    # ...
    ax=ax_q1
)

### Question 2: Relation entre l'âge et le mouvement

Votre superviseure se demande s'il existe une relation entre l'âge des participant.e.s (`age`) et les niveaux de mouvement (`MeanFD`) lors des acquisitions d'IRMf. Pour vous permettre de vous faire une idée sur la question, elle vous demande de visualiser cette relation. Plus précisément, vous devrez:

- Choisir la fonction adéquate de la librairie `seaborn` vous permettant de visualiser uniquement la relation entre les deux variables, tout en y incluant un modèle de régression.
- Spécifier la structure de données (dataframe) à utiliser dans la fonction.
- L'âge doit être traité comme votre variable indépendante.
- Le mouvement doit être traité comme variable dépendante.

In [ ]:
ax_q2 = plt.subplot()

# Modifiez les lignes ci-dessous. Une fois que vous aurez remplacé `<choisissez la bonne fonction>` par la bonne fonction
# et que vous aurez remplacé les `...` par les bonnes variables, décommenter les lignes suivantes:

#sns.<choisissez la bonne fonction>(
#    ...,
#    ax=ax_q2
#)

### Question 3: Valeurs aberrantes et numéro de participant.e

Vous effectuez la figure montrant la relation entre l'âge et le mouvement et vous remarquez qu'il semble y avoir un.e participant.e ayant une valeur de FD moyenne potentiellement aberrante. Avant de rencontrer votre superviseure pour lui montrer votre figure, vous aimeriez pouvoir extraire le numéro de participant.e directement à partir de celle-ci. 

Reproduisez la figure que vous avez faite à la question précédente avec `plotly.express`. Pour cette question, la fonction de plotly est indiquée. Lorsque vous allez modifier puis rouler la cellule ci-dessous, vous devrez être en mesure de voir les valeurs des variables `age`, `MeanFD` et `Subject` lorsque vous placez votre curseur au-dessus d'un point.

&#128161; Pour visualiser la régression, utilisez la méthode des moindres carrés ordinaires.

In [ ]:
q3 = px.scatter(
    # ...
)

q3.show()

#### Question 3: Valeurs aberrantes et numéro de participant.e (suite)

Maitenant que vous pouvez avoir accès au numéro de participant.e directement à partir de votre figure, stocker ce numéro dans la variable `q3_outlier` dans la cellule ci-dessous.

In [ ]:
# REMPLACEZ LES ... PAR VOTRE RÉPONSE
q3_outlier = ...
print(f'Le sujet {q3_outlier} semble avoir une valeur de FD moyenne aberrante.')

### Question 4: Effet du site

En prévision de l'interprétation du modèle d'apprentissage machine. Votre superviseure se démande si certains facteurs pourraient confondre les prédictions du groupe (`adhd`). Notamment, elle se demande si le site d'acquisition (`site`) pourrait être une variable confondante.

Complétez la cellule ci-dessous en y ajoutant les paramètres suivants:
- La taille de votre figure devrait avoir une largeur de 10 et une hauteur de 3.
- Visualisez les sites sur l'axe des x
- Le nombre de participant.e.s par site doit être séparé par groupe
- Modifiez le nom de l'axe des y pour celui stocké dans la variable `y_axis_name` dans la cellule ci-dessous. &#128161; Vous devrez modifier un attribut de la variable `ax_q4`.

Modifiez les `...` dans la cellule ci-dessous selon ce qui a été demandé.

In [ ]:
# Spécifiez la taille de la figure
fig_q4, ax_q4 = plt.subplots(1, ...)

# Spécifiez les paramètres demandés dans la fonction ci-dessous
sns.countplot(
    ...,
    ax=ax_q4
)

# Modifiez le nom de l'axe
y_axis_name = 'Nombre de participants' # NE PAS MODIFIER
ax_q4. ...

# Partie 2 : Exploration des données d'IRM fonctionnelle

Dans cette section, vous allez explorer les données d'IRM fonctionnelle du jeu de données ADHD200.

In [ ]:
# Chargement de l'atlas BASC 12 ROIs — ne pas modifier
basc = datasets.fetch_atlas_basc_multiscale_2015(
    data_dir=data_dir, resolution=12
)
labels = basc.labels
atlas_img = basc.maps
print(f"Atlas chargé : {atlas_img}")
print(f"Nombre de régions: {len(labels)}")

### Question 5: Visualiser l'atlas

Visualisez les régions d'intérêt (*Regions of Interest*; ROIs) grâce à la fonction `plot_roi` de la librairie `nilearn`. Visualisez l'atlas selon les instructions suivantes:

- Enlevez la croix montrant la position des coordonnées pour chaque coupe
- Ajoutez le titre `Atlas BASC - 12 ROIs`
- Montrez l'atlas à la coordonnées (0, 0, 0)

&#128161; Consultez la documentation de la fonction [`plot_roi`](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_roi.html) pour déterminer quels paramètres inclure dans la fonction.

In [ ]:
title_q5 = 'Atlas BASC - 12 ROIs'

fig_q5 = plotting.plot_roi(
    ...
)

### Question 6 - Séries temporelles

Comparez les signaux temporelles de la première région cérébrale de l'atlas entre le sujet ayant le plus de mouvement et celui ayant le plus de mouvement (voir figure générée à la question 3). Plus spécifiquement, votre superviseure vous demande de faire une figure avec 1 ligne et deux colonnes. Dans la colonne de gauche, vous devrez visualisez votre région d'intérêt. Dans la colonne de droite, vous devrez visualiser la série temporelle pour vos deux sujets.

**Colonne de gauche:**
- Visualisez votre région d'intérêt grâce à la fonction `plot_roi` de `nilearn`
- Enlevez les annotations
- Enlevez la barre de couleur (`colorbar`)
- Visualisez uniquement la coupe axiale selon les coordonnées spécifiées dans la variable `coords` dans la cellule ci-dessous
- N'oubliez pas de spécifier votre axe dans le sous-graphique !

&#128161; Consultez la documentation de la fonction [`plot_roi`](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_roi.html) pour déterminer quels paramètres inclure dans la fonction.

**Colonne de droite:**
- Visualisez les séries temporelles de vos deux sujets pour votre région d'intérêt grâce à la fonction `plot` de `matplotlib`
- Spécifiez les suivantes étiquettes pour chacun de vos sujets: `'low'` pour votre sujet avec le moins de mouvement et `'high'` pour votre sujet avec le plus de mouvement
- Insérez une légende en bas à gauche de votre sous-graphique pour indiquer quelle couleur appartient à quel sujet.
- La série temporelle pour le sujet ayant le plus de mouvement (`'high'`) devrait être en pointillé (`'--'`)
- Ajoutez le titre `'Volumes'` à votre axe des x et metter la taille de la police pour ce titre à `14`
- L'épaisseur de vos ligne devrait être de `3`.

&#128161; Référez vous au tutoriel du cours de visualisation et à [la documentation de matplotlib](https://matplotlib.org/stable/) au besoin.

In [ ]:
# Créez le layout de votre figure
fig_q6, axes_q6 = plt.subplots(1, 2, figsize=(20, 5), width_ratios=[1, 3]) # NE PAS MODIFIER

# Indiquez les numéros de sujet
id_low_q6 = ... # À MODIFIER
id_high_q6 = ... # À MODIFIER

# Allez chercher les données des sujets spécifiés plus haut
img_low_q6 = [f for f in func if str(id_low_q6) in f][0] # NE PAS MODIFIER
img_high_q6 = [f for f in func if str(id_high_q6) in f][0] # NE PAS MODIFIER

# Instanciez le masque
# NE PAS MODIFIER
mask = NiftiLabelsMasker(
    labels_img=atlas_img,
    labels = labels
)
# Extrayez les séries temporelles pour les sujets spécifiés plus haut
# NE PAS MODIFIER
timeserie_low_q6=mask.fit_transform(
    img_low_q6
)
timeserie_high_q6=mask.fit_transform(
    img_high_q6
)

# Allez chercher les séries temporelles pour la première région d'intérêt
roi_id_q6 = ... # À MODIFIER
roi_low_q6 = timeserie_low_q6[:, roi_id_q6] # NE PAS MODIFIER
roi_high_q6 = timeserie_high_q6[:, roi_id_q6] # NE PAS MODIFIER

# Sélectionnez la région spécifique dans l'atlas
roi_atlas = math_img('img1==1', img1=atlas_img) # NE PAS MODIFIER

# Ajoutez la région d'intérêt à la figure
coords = (0,)

# À MODIFIER: SPÉCIFIEZ LES PARAMÈTRES REQUIS DANS LA FONCTION CI-DESSOUS
plotting.plot_roi(
    ...
)

# Ajoutez les séries temporelles à la figure
# À MODIFIER: SPÉCIFIEZ L'AXE, LA FONCTION ET LES PARAMÈTRES REQUIS DANS LA FONCTION POUR LE PARTICIPANT AYANT LE MOINS DE MOUVEMENT
axes_q6...
# À MODIFIER: SPÉCIFIEZ L'AXE, LA FONCTION ET LES PARAMÈTRES REQUIS DANS LA FONCTION POUR LE PARTICIPANT AYANT LE PLUS DE MOUVEMENT
axes_q6...
# À MODIFIER: AJOUTEZ LA LÉGENDE ET LES PARAMÈTRES NÉCESSAIRES
axes_q6...
# À MODIFIER: APPORTEZ LES MODIFICATIONS DEMANDÉES À L'AXE DES X
axes_q6...

### Question 7 - Carpet plot

Votre superviseure apprécie beaucoup la figure que vous avez créé précédemment. Par contre, elle aimerait pouvoir comparer les séries temporelles de tous les voxels (et non simplement pour une région d'intérêt) pour ces deux sujets. Pour ce faire, vous devrez utiliser la fonction `plot_carpet` de `nilearn`. Plus précisément, vous allez devoir:

- Créez une figure avec deux rangées (1 colonne)
- La rangée du haut devra contenir le *carpet plot* pour le sujet ayant le moins de mouvement (plus petite valeur de FD moyenne; voir question précédente).
- La rangée du bas devra contenir le *carpet plot* pour le sujet ayant le plus de mouvement (plus grande valeur de FD moyenne; voir question précédente).
- Pour chaque sous-figure, vous devrez ajouter un titre. Pour la figure du haut, vous devrez utiliser comme titre 'Participant with the lowest Mean FD value'. Pour la figure du bas, vous devrez utiliser comme titre 'Participant with the highest Mean FD value'
- Séparez le *carpet plot* selon les régions de votre atlas. **Indice:** vous allez devoir utiliser les paramètres `mask_img` et `mask_labels` de la fonction `plot_carpet`.

&#128161; Les données à visualiser ont déjà étaient stockées dans des variables à la question précédente. Pour déterminer quelle variable utilisée pour le masque, regardez attentivement le type de la variable attendue dans [la documentation de nilearn](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_carpet.html).


In [ ]:
labels_dict_q7 = {item: int(item) for item in labels if item != 'Background'} # NE PAS MODIFIER

fig_q7, axes_q7 = plt.subplots(2, 1, figsize=(16, 12))

display_low_q7 = plotting.plot_carpet(
    ...
)
display_high_q7 = plotting.plot_carpet(
    ...
)

# Partie 3 :  Interprétation des modèles d'apprentissage machine

Dans cette partie, nous allons utiliser quelques outputs provenant de l'exercice du cours d'apprentissage machine pour la neuroimagerie. Les données sont déjà fournies.

In [ ]:
# NE PAS MODIFIER
# Chargez les poids du modèle
coef_svm = np.load('exercice_ml_coefficients.npz')['coef']

# Chargez les séries temporelles
timeseries = np.load('exercice_ml_timeseries.npz', allow_pickle=True)['timeseries'].tolist()

# Instanciez un objet ConnectivityMeasure
correlation_measure = ConnectivityMeasure(
    kind='correlation',
    vectorize=True,
    discard_diagonal=True
)
correlation_measure.fit_transform(timeseries)

### Question 8: Matrice de corrélation - partie 1

À partir des données que nous avons téléchargées dans la cellule ci-dessus, visualisez les poids (coefficients) du modèle sous forme de matrice selon les instructions suivantes:
- Visualisez uniquement la partie inférieure de la matrice
- Utilisez la palette de couleur `vik` de la librairie `cmcrameri` en utilisant le module `cm`
- Spécifiez les étiquettes (de 0 à 12)
- Ajoutez un titre

In [ ]:
# Appliquez une transformation pour obtenir les poids du modèle sous forme de matrice
matrice_coef = ...

# Visualisez la matrice
display_q8 = plotting.plot_matrix(
    ...
)

### Question 9: Matrice de corrélation - partie 2

Prenez le temps d'observer la matrice que vous venez de créer. Remarquez-vous quelque chose au niveau de la barre de couleur ?

Modifiez la barre de couleur pour faire en sorte quelle soit symétrique. Réutilisez les mêmes paramètres ceux utilisés à la question précédente, mais apportez la modification suivante:
- Les extrémités de la barre de couleur devrait avoir comme valeur absolue la valeur minimale de `matrice_coef`

&#128161; Vous devrez modifier le paramètre nécessaire dans la fonction `plot_matrix`. Ce paramètre n'est pas explicitement listé dans la documentation de la fonction `plot_matrix`, mais vous pouvez vous fier sur [la documentation de la fonction `plot_img`](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.plot_img.html#nilearn.plotting.plot_img).

In [ ]:
# Sauvegardez la valeur minimale absolue de la matrice
# Indice: vous devrez utiliser la fonction `np.abs()` de la librairie numpy
extremite_q9 = ...

# Visualisez la matrice
display_q9 = plotting.plot_matrix(
    ...
)

### Question 10: Connectome

Vous aimeriez trouver une façon un peu plus intuitive de visualiser les poids de votre modèle. Votre superviseure vous suggère de les visualiser sous forme de connectome 3D. Spécifiquement, vous devrez:
1. Définir les coordonnées de vos régions d'intérêts (ROIs) dans la variable `coords_q10`.
2. Remplacer les éléments de la diagonale de la matrice `matrice_coef` à 0.
3. Visualiser le connectome.

Pour le connectome, vous devrez:
- Modifier le taille des noeuds (*node*) à `5`
- Modifier la taille de la police de la barre de couleur à `14`
- Ajouter un titre
- Modifier la taille de la police du titre à `20`

In [ ]:
# Allez chercher les coordonnées de vos régions
coords_q10 = ...

# Mettez les éléments de la diagonale de la matrice à 0
...

# Visualisez le connectome en 3D
viewer_q10 = plotting.view_connectome(
    ...
)

viewer_q10

### Question de réflexion (non notée)

Y a-t-il quelque chose d'inattendue dans votre visualisation ? Selon vous que s'est-il passé ?